# E-Commerce Product Sales & Demand Analysis

**Project by:** Yamini

This notebook performs data loading, cleaning, exploratory data analysis (EDA), visualization, and demand classification using a Random Forest model.

In [ ]:
# Import required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

sns.set_theme(style="whitegrid")


## 1. Load the Dataset

Place `ecommerce_dataset.csv` in the same folder as this notebook.

In [ ]:
df = pd.read_csv("ecommerce_dataset.csv")
print("Dataset shape:", df.shape)
display(df.head())


## 2. Basic Dataset Information

In [ ]:
display(df.info())
display(df.describe(include="all").T)


## 3. Data Cleaning

We check duplicate records, missing values, and remove duplicate rows if any are present.

In [ ]:
print("Missing values:")
display(df.isnull().sum())

print("Duplicate rows:", df.duplicated().sum())

df = df.drop_duplicates().copy()

print("Shape after cleaning:", df.shape)


## 4. Category-wise Sales Analysis

In [ ]:
category_sales = df.groupby("Category")["Units_Sold"].sum().sort_values(ascending=False)
display(category_sales.to_frame("Total_Units_Sold"))

plt.figure(figsize=(9,5))
category_sales.plot(kind="bar")
plt.title("Total Units Sold by Category")
plt.xlabel("Category")
plt.ylabel("Units Sold")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()


## 5. Average Product Rating by Category

In [ ]:
avg_rating = df.groupby("Category")["Rating"].mean().sort_values(ascending=False)
display(avg_rating.to_frame("Average_Rating"))

plt.figure(figsize=(9,5))
avg_rating.plot(kind="bar")
plt.title("Average Rating by Category")
plt.xlabel("Category")
plt.ylabel("Average Rating")
plt.ylim(0,5)
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()


## 6. Price vs Units Sold

In [ ]:
plt.figure(figsize=(9,5))
sns.scatterplot(data=df, x="Price_INR", y="Units_Sold", hue="Category")
plt.title("Price vs Units Sold")
plt.xlabel("Price (INR)")
plt.ylabel("Units Sold")
plt.tight_layout()
plt.show()


## 7. Correlation Analysis

In [ ]:
numeric_cols = ["Price_INR","Rating","Reviews","Units_Sold","Discount_Percent"]
corr = df[numeric_cols].corr()

plt.figure(figsize=(8,6))
sns.heatmap(corr, annot=True, fmt=".2f")
plt.title("Correlation Matrix")
plt.tight_layout()
plt.show()


## 8. Demand Distribution

In [ ]:
demand_counts = df["Demand"].value_counts()
display(demand_counts.to_frame("Product_Count"))

plt.figure(figsize=(7,5))
demand_counts.plot(kind="bar")
plt.title("Demand Distribution")
plt.xlabel("Demand")
plt.ylabel("Number of Products")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()


## 9. Machine Learning: Demand Classification

The target variable is `Demand` with three classes: Low, Medium, and High. The model uses product category, price, rating, reviews, units sold, discount, and stock status.

In [ ]:
X = df.drop(columns=["Demand"])
y = df["Demand"]

categorical_features = ["Category", "Product_Name", "Brand", "In_Stock"]
numeric_features = ["Price_INR", "Rating", "Reviews", "Units_Sold", "Discount_Percent"]

preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features),
        ("num", "passthrough", numeric_features)
    ]
)

model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("classifier", RandomForestClassifier(
        n_estimators=150,
        random_state=42,
        class_weight="balanced"
    ))
])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

model.fit(X_train, y_train)
y_pred = model.predict(X_test)

print("Accuracy:", round(accuracy_score(y_test, y_pred), 4))
print("\nClassification Report:")
print(classification_report(y_test, y_pred))


## 10. Confusion Matrix

In [ ]:
cm = confusion_matrix(y_test, y_pred, labels=["Low","Medium","High"])

plt.figure(figsize=(7,5))
sns.heatmap(cm, annot=True, fmt="d", xticklabels=["Low","Medium","High"],
            yticklabels=["Low","Medium","High"])
plt.title("Demand Classification Confusion Matrix")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.tight_layout()
plt.show()


## 11. Sample Prediction

In [ ]:
sample = pd.DataFrame([{
    "Product_ID": 9999,
    "Category": "Electronics",
    "Product_Name": "Wireless Headphones",
    "Brand": "Boat",
    "Price_INR": 1499,
    "Rating": 4.4,
    "Reviews": 1200,
    "Units_Sold": 135,
    "Discount_Percent": 15,
    "In_Stock": "Yes"
}])

prediction = model.predict(sample)[0]
print("Predicted Demand:", prediction)


## 12. Conclusion

The analysis identifies category-wise sales patterns, rating trends, price-sales relationships, and demand levels. A Random Forest classifier is also demonstrated for educational demand classification. The dataset is synthetic and intended for academic/project demonstration, not for real business decisions.